<a href="https://colab.research.google.com/github/shinmyatnoezin/HousePredictionProject/blob/main/HousePrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import Libraries

In [ ]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import SplineTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error,
    median_absolute_error,
    explained_variance_score,
    max_error
)

sns.set_style("whitegrid")


# 2. Load HM Land Registry Data

In [ ]:
# =========================
# 2. LOAD HM LAND REGISTRY DATA
# =========================
files = glob.glob("data/*.csv")

if len(files) == 0:
    raise FileNotFoundError("No CSV files found in the 'data' folder.")

df_list = []
for file in files:
    temp = pd.read_csv(file, header=None, low_memory=False)
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

df.columns = [
    "transaction_id", "price", "date", "postcode",
    "property_type", "new_build", "tenure",
    "paon", "saon", "street", "locality",
    "town_city", "district", "county",
    "ppd_category", "record_status"
]

print("Land Registry data shape:", df.shape)

# 3. Basic Cleaning

In [ ]:
# =========================
# 3. BASIC CLEANING
# =========================
df = df[[
    "price", "date", "postcode", "property_type",
    "new_build", "tenure", "town_city", "district", "county"
]].copy()

text_cols = ["postcode", "property_type", "new_build", "tenure", "town_city", "district", "county"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.dropna(subset=["price", "date", "postcode"])
df = df[df["price"] > 0]

print("After basic cleaning:", df.shape)

# 4. Load UK Postcodes Data

In [ ]:
# =========================
# 4. LOAD UK POSTCODES DATA
# =========================
postcode_file = "ukpostcodes.csv"
postcode_df = pd.read_csv(postcode_file, low_memory=False)

required_cols = ["postcode", "latitude", "longitude"]
for col in required_cols:
    if col not in postcode_df.columns:
        raise KeyError(f"Column '{col}' not found in postcode dataset.")

postcode_df = postcode_df[["postcode", "latitude", "longitude"]].copy()
postcode_df["postcode"] = postcode_df["postcode"].astype(str).str.strip()
postcode_df["latitude"] = pd.to_numeric(postcode_df["latitude"], errors="coerce")
postcode_df["longitude"] = pd.to_numeric(postcode_df["longitude"], errors="coerce")

postcode_df = postcode_df.dropna(subset=["postcode", "latitude", "longitude"])
postcode_df = postcode_df.drop_duplicates(subset=["postcode"])

print("Postcode data shape:", postcode_df.shape)

# 5. Merge Land Registry + Postcodes

In [ ]:
# =========================
# 5. MERGE DATASETS
# =========================
df = df.merge(postcode_df, on="postcode", how="left")

print("After merge:", df.shape)
print("Missing latitude:", df["latitude"].isna().sum())
print("Missing longitude:", df["longitude"].isna().sum())

df = df.dropna(subset=["latitude", "longitude"]).copy()

print("After dropping missing coordinates:", df.shape)

# 6. Filter To A Specific Area

In [ ]:
# =========================
# 6. FILTER FOR BRISTOL
# =========================
df = df[df["district"].str.contains("BRISTOL", case=False, na=False)].copy()
print("After Bristol filter:", df.shape)

# 7. Handle Outliers

In [ ]:
# =========================
# 7. HANDLE OUTLIERS
# =========================
# Remove extreme top 1% prices to make model more stable
upper_limit = df["price"].quantile(0.99)
df = df[df["price"] <= upper_limit].copy()

print("After removing top 1% price outliers:", df.shape)

# 8. Create Extra Location-Relate Features

**7.1 Time Features**

In [ ]:
# =========================
# 7. CREATE FEATURES (NO TARGET USED)
# =========================

# Time features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

**7.2 Bristol Approximate Center Reference Point**

In [ ]:
# 7.2Reference point: Bristol city centre
reference_lat = 51.4545
reference_lon = -2.5879

# Haversine distance in km
lat1 = np.radians(df["latitude"])
lon1 = np.radians(df["longitude"])
lat2 = np.radians(reference_lat)
lon2 = np.radians(reference_lon)

dlat = lat2 - lat1
dlon = lon2 - lon1

a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
R = 6371  # Earth radius in km

df["distance_to_city_center"] = R * c

**7.3 Interaction Feature**

In [ ]:
# 7.3 Interaction feature
df["lat_long_interaction"] = df["latitude"] * df["longitude"]

**7.4 Log Price**

In [ ]:
# 7.4 Log price target
df["log_price"] = np.log1p(df["price"])

print("Safe features created.")

# 9. EDA Analysis

**9.1 Dataset overview**

In [ ]:
# =========================
# 9. EDA ANALYSIS
# =========================

# 9.1 Dataset overview
print("\nData types:\n", df.dtypes)
print("\nSummary statistics:\n", df.describe(include="all"))

**9.2 Missing Values**

In [ ]:
# 9.2 Missing values
print("\nMissing values:\n", df.isnull().sum())

**9.3 Distribution Of Price**

In [ ]:
# 9.3 Distribution of house price
plt.figure(figsize=(10, 5))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

**9.4 Log Price Distribution**

In [ ]:
# Distribution of log house prices
plt.figure(figsize=(10, 5))
sns.histplot(df["log_price"], bins=50, kde=True)
plt.title("Distribution of Log House Prices")
plt.xlabel("Log Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

**9.5 Scatter Plot Of Longitude vs Latitude Colored By Price**

In [ ]:
# 9.5 Scatter plot of longitude vs latitude colored by price
sample_df = df.sample(min(5000, len(df)), random_state=42)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    sample_df["longitude"],
    sample_df["latitude"],
    c=sample_df["price"],
    s=10,
    alpha=0.7
)
plt.colorbar(scatter, label="Price")
plt.title("House Prices by Geographic Location")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

**9.6 Boxplot By Property Type**

In [ ]:
# 9.6 Boxplot by property type
plt.figure(figsize=(8, 5))
sns.boxplot(x="property_type", y="price", data=df)
plt.title("House Price by Property Type")
plt.xlabel("Property Type")
plt.ylabel("Price")
plt.tight_layout()
plt.show()

**9.7 Transactions Per Year**

In [ ]:
# 9.7 Transactions per year
plt.figure(figsize=(8, 5))
sns.countplot(x="year", data=df, order=sorted(df["year"].dropna().unique()))
plt.title("Number of Transactions per Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


**9.8 Average Price By Year**

In [ ]:
# 9.8 Average price by year
avg_price_year = df.groupby("year")["price"].mean().reset_index()

plt.figure(figsize=(8, 5))
plt.plot(avg_price_year["year"], avg_price_year["price"], marker="o")
plt.title("Average House Price by Year")
plt.xlabel("Year")
plt.ylabel("Average Price")
plt.tight_layout()
plt.show()

# 10. Prepare Features & Train/Test Split

In [ ]:
# =========================
# 10. Prepare Features & Train/Test Split
# =========================
base_cols = [
    "latitude", "longitude",
    "distance_to_city_center",
    "lat_long_interaction",
    "year", "month",
    "property_type", "new_build", "tenure",
    "district", "town_city"
]

X = df[base_cols].copy()
y = df["log_price"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

X_train = X_train.copy()
X_test = X_test.copy()


# 11. Fit Location Transforms On Train Only

**11.1 Latitude and Longitude bins using train only**

In [ ]:
# =========================
# 11. FIT LOCATION TRANSFORMS ON TRAIN ONLY
# =========================

# 11.1 Latitude and longitude bins using train only
lat_bin_edges = np.linspace(X_train["latitude"].min(), X_train["latitude"].max(), 11)
lon_bin_edges = np.linspace(X_train["longitude"].min(), X_train["longitude"].max(), 11)

lat_bin_edges[-1] += 1e-9
lon_bin_edges[-1] += 1e-9

X_train["lat_bin"] = pd.cut(
    X_train["latitude"], bins=lat_bin_edges, labels=False, include_lowest=True
)
X_test["lat_bin"] = pd.cut(
    X_test["latitude"], bins=lat_bin_edges, labels=False, include_lowest=True
)

X_train["lon_bin"] = pd.cut(
    X_train["longitude"], bins=lon_bin_edges, labels=False, include_lowest=True
)
X_test["lon_bin"] = pd.cut(
    X_test["longitude"], bins=lon_bin_edges, labels=False, include_lowest=True
)

X_train["lat_bin"] = X_train["lat_bin"].fillna(-1).astype(int)
X_train["lon_bin"] = X_train["lon_bin"].fillna(-1).astype(int)
X_test["lat_bin"] = X_test["lat_bin"].fillna(-1).astype(int)
X_test["lon_bin"] = X_test["lon_bin"].fillna(-1).astype(int)


**11.2 KMeans clustering on training coordinates only**

In [ ]:
# 11.2 KMeans clustering on training coordinates only
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
X_train["location_cluster"] = kmeans.fit_predict(X_train[["latitude", "longitude"]])
X_test["location_cluster"] = kmeans.predict(X_test[["latitude", "longitude"]])

**11.3 Spline Features For Latitude And Longitude**

In [ ]:
#11.3 Spline Features For Latitude And Longitude
lat_spline = SplineTransformer(degree=3, n_knots=5, include_bias=False)
lon_spline = SplineTransformer(degree=3, n_knots=5, include_bias=False)

lat_train_spline = lat_spline.fit_transform(X_train[["latitude"]])
lat_test_spline = lat_spline.transform(X_test[["latitude"]])

lon_train_spline = lon_spline.fit_transform(X_train[["longitude"]])
lon_test_spline = lon_spline.transform(X_test[["longitude"]])

lat_spline_cols = [f"lat_spline_{i}" for i in range(lat_train_spline.shape[1])]
lon_spline_cols = [f"lon_spline_{i}" for i in range(lon_train_spline.shape[1])]

lat_train_spline_df = pd.DataFrame(lat_train_spline, columns=lat_spline_cols, index=X_train.index)
lat_test_spline_df = pd.DataFrame(lat_test_spline, columns=lat_spline_cols, index=X_test.index)

lon_train_spline_df = pd.DataFrame(lon_train_spline, columns=lon_spline_cols, index=X_train.index)
lon_test_spline_df = pd.DataFrame(lon_test_spline, columns=lon_spline_cols, index=X_test.index)

X_train = pd.concat([X_train, lat_train_spline_df, lon_train_spline_df], axis=1)
X_test = pd.concat([X_test, lat_test_spline_df, lon_test_spline_df], axis=1)

print("Added spline features:")
print(lat_spline_cols + lon_spline_cols)

# 12. Evaluation Function

In [ ]:
# =========================
# 12. Evaluation Function
# =========================
def evaluate_predictions(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    maxerr = max_error(y_true, y_pred)

    y_true_safe = np.where(y_true == 0, 1, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100
    smape = 100 * np.mean(
        2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-10)
    )

    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MedianAE": medae,
        "MAPE": mape,
        "SMAPE": smape,
        "ExplainedVariance": evs,
        "MaxError": maxerr
    }


# 13. Model Runner

In [ ]:
# =========================
# 13. Model Runner
# =========================
categorical_cols = ["property_type", "new_build", "tenure", "district", "town_city"]

def run_experiment(model_name, feature_cols):
    """
    feature_cols: columns to include before encoding categoricals
    """
    X_train_exp = X_train[feature_cols].copy()
    X_test_exp = X_test[feature_cols].copy()

    # figure out which selected features are categorical
    selected_cats = [col for col in categorical_cols if col in X_train_exp.columns]

    # one-hot encode only selected categorical columns
    X_train_exp = pd.get_dummies(X_train_exp, columns=selected_cats, drop_first=True)
    X_test_exp = pd.get_dummies(X_test_exp, columns=selected_cats, drop_first=True)

    # align columns
    X_train_exp, X_test_exp = X_train_exp.align(X_test_exp, join="left", axis=1, fill_value=0)

    # model
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_exp, y_train)
    pred_log = model.predict(X_test_exp)

    results = evaluate_predictions(y_test, pred_log)
    results["Model"] = model_name
    results["Num_Features"] = X_train_exp.shape[1]

    return results, model, X_train_exp.columns.tolist()


# 14. Define Feature Sets

In [ ]:
# =========================
# 14. DEFINE FEATURE SETS
# =========================
feature_sets = {
    "Model 1 - Raw Coordinates": [
        "latitude", "longitude"
    ],

    "Model 2 - Raw + Distance": [
        "latitude", "longitude",
        "distance_to_city_center"
    ],

    "Model 3 - Clean Full (No Spline)": [
        "latitude", "longitude",
        "distance_to_city_center",
        "lat_long_interaction",
        "lat_bin", "lon_bin",
        "location_cluster",
        "year", "month",
        "property_type", "new_build", "tenure",
        "district", "town_city"
    ],

    "Model 4 - Spline Only": [
        *lat_spline_cols,
        *lon_spline_cols
    ],

    "Model 5 - Spline Full": [
        "latitude", "longitude",
        *lat_spline_cols,
        *lon_spline_cols,
        "distance_to_city_center",
        "lat_long_interaction",
        "lat_bin", "lon_bin",
        "location_cluster",
        "year", "month",
        "property_type", "new_build", "tenure",
        "district", "town_city"
    ]
}

# 15. Run All Results

In [ ]:
# =========================
# 15. Run All Results
# =========================

all_results = []
trained_models = {}
model_columns = {}

for model_name, features in feature_sets.items():
    print(f"Running {model_name} ...")
    results, model, used_columns = run_experiment(model_name, features)
    all_results.append(results)
    trained_models[model_name] = model
    model_columns[model_name] = used_columns

results_df = pd.DataFrame(all_results)

# reorder columns
results_df = results_df[[
    "Model", "Num_Features", "R2", "RMSE", "MAE",
    "MedianAE", "MAPE", "SMAPE", "ExplainedVariance", "MaxError"
]]

# sort by R2 descending
results_df = results_df.sort_values(by="R2", ascending=False).reset_index(drop=True)

print("\n===== FEATURE COMPARISON RESULTS =====")
print(results_df)

# 16. Visualise Comparsion

In [ ]:
# =========================
# 16. Visualise Comparison
# =========================
plt.figure(figsize=(12, 6))
plt.bar(results_df["Model"], results_df["R2"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("R²")
plt.title("Model Comparison by R²")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
plt.bar(results_df["Model"], results_df["MAPE"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("MAPE (%)")
plt.title("Model Comparison by Relative Error (MAPE)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("RMSE")
plt.title("Model Comparison by RMSE")
plt.tight_layout()
plt.show()


# 17. Save Results Table

In [ ]:
# =========================
# 17. SAVE RESULTS TABLE
# =========================
results_df.to_csv("feature_comparison_results.csv", index=False)
print("\nSaved: feature_comparison_results.csv")

## 18. Best Model


In [ ]:
# =========================
# 18. Best Model
# =========================
best_model_name = results_df.iloc[0]["Model"]
print("\nBest model based on R²:", best_model_name)
print(results_df.iloc[0])


# 19. Top Features Of Best Model

In [ ]:
# =========================
# 19. TOP FEATURES OF BEST MODEL
# =========================
best_model = trained_models[best_model_name]
best_columns = model_columns[best_model_name]

importances = pd.Series(best_model.feature_importances_, index=best_columns).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.head(15).sort_values().plot(kind="barh")
plt.title(f"Top 15 Feature Importances - {best_model_name}")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\nTop 15 important features in best model:")
print(importances.head(15))

# 20. Save Best Model Predictions

In [ ]:
# =========================
# 20. SAVE BEST MODEL PREDICTIONS
# actual price / predicted price / residual / percentage error
# =========================

def get_model_predictions(model_name, feature_cols):
    X_train_exp = X_train[feature_cols].copy()
    X_test_exp = X_test[feature_cols].copy()

    selected_cats = [col for col in categorical_cols if col in X_train_exp.columns]

    X_train_exp = pd.get_dummies(X_train_exp, columns=selected_cats, drop_first=True)
    X_test_exp = pd.get_dummies(X_test_exp, columns=selected_cats, drop_first=True)

    X_train_exp, X_test_exp = X_train_exp.align(
        X_test_exp, join="left", axis=1, fill_value=0
    )

    model = trained_models[model_name]
    pred_log = model.predict(X_test_exp)

    actual_price = np.expm1(y_test)
    predicted_price = np.expm1(pred_log)

    results_pred = pd.DataFrame({
        "actual_price": actual_price,
        "predicted_price": predicted_price
    }, index=y_test.index)

    results_pred["residual"] = results_pred["actual_price"] - results_pred["predicted_price"]
    results_pred["absolute_error"] = np.abs(results_pred["residual"])
    results_pred["percentage_error"] = (
        results_pred["absolute_error"] / results_pred["actual_price"].replace(0, np.nan)
    ) * 100

    return results_pred


best_predictions_df = get_model_predictions(
    best_model_name,
    feature_sets[best_model_name]
)

best_predictions_df.to_csv("best_model_predictions.csv", index=False)
print("\nSaved: best_model_predictions.csv")
print(best_predictions_df.head())

# 21. Actual VS Predicted Plot for Model 4

In [ ]:
# =========================
# 21. ACTUAL VS PREDICTED PLOT FOR MODEL 4
# =========================
plt.figure(figsize=(8, 8))
plt.scatter(
    best_predictions_df["actual_price"],
    best_predictions_df["predicted_price"],
    alpha=0.5
)

min_val = min(best_predictions_df["actual_price"].min(), best_predictions_df["predicted_price"].min())
max_val = max(best_predictions_df["actual_price"].max(), best_predictions_df["predicted_price"].max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title(f"Actual vs Predicted House Prices - {best_model_name}")
plt.tight_layout()
plt.show()

# 22. Residual Plot for Model 4

In [ ]:
# =========================
# 22. RESIDUAL PLOT FOR MODEL 4
# =========================
plt.figure(figsize=(10, 6))
plt.scatter(
    best_predictions_df["predicted_price"],
    best_predictions_df["residual"],
    alpha=0.5
)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title(f"Residual Plot - {best_model_name}")
plt.tight_layout()
plt.show()

# 24. Clean Model Predictions

In [ ]:
# =========================
# 24. CLEAN MODEL (MODEL 3) PREDICTIONS
# =========================

clean_model_name = "Model 3 - Clean Full (No Spline)"

clean_predictions_df = get_model_predictions(
    clean_model_name,
    feature_sets[clean_model_name]
)

clean_predictions_df.to_csv("clean_model_predictions.csv", index=False)
print("\nSaved: clean_model_predictions.csv")
print(clean_predictions_df.head())

# 25. Clean Model - Actual VS Predicted

In [ ]:
# =========================
# 25. CLEAN MODEL - ACTUAL VS PREDICTED
# =========================
plt.figure(figsize=(8, 8))

plt.scatter(
    clean_predictions_df["actual_price"],
    clean_predictions_df["predicted_price"],
    alpha=0.5
)

min_val = min(
    clean_predictions_df["actual_price"].min(),
    clean_predictions_df["predicted_price"].min()
)

max_val = max(
    clean_predictions_df["actual_price"].max(),
    clean_predictions_df["predicted_price"].max()
)

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted - Clean Model (No Spline)")
plt.tight_layout()
plt.show()

# 26. Clean Model - Residuals VS Actual

In [ ]:
# =========================
# 26. CLEAN MODEL - RESIDUALS VS ACTUAL
# =========================
plt.figure(figsize=(10, 6))

plt.scatter(
    clean_predictions_df["actual_price"],
    clean_predictions_df["residual"],
    alpha=0.5
)

plt.axhline(y=0, linestyle="--")

plt.xlabel("Actual Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residuals vs Actual Price - Clean Model")
plt.tight_layout()
plt.show()